# 05 · E7 Autoencoder early stopping (ULB)

In the reference protocol the AE stops training when its loss on the validation partition stops improving. The same validation partition is then used to set the threshold. This notebook checks whether that sharing matters by retraining only the AE with stopping on 10% of the legitimate training rows held out for the purpose.

IF and OCSVM scores, and the shared-validation AE, are taken from notebook 01, so the only thing that differs between the two protocols is the AE's stopping data.

| | |
|---|---|
| Trains models | yes: one AE per seed |
| Expected time | about 5 min on a GPU runtime |
| Needs | notebook 01 |
| Writes | `results/E7_early_stopping/`: `early_stopping_by_seed.csv`, `early_stopping_summary.csv` |

In [ ]:
import os, sys

ROOT = os.environ.get("RAREGUARD_ROOT", "/content/drive/MyDrive/PhD/RAREGuard_v6")

if ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

sys.path.insert(0, os.path.join(ROOT, "code"))
import rareguard_lib as rg

try:
    from IPython.display import display
except ImportError:
    display = print

cfg = rg.load_config(ROOT)
env = rg.show_environment()

In [ ]:
import numpy as np, pandas as pd

EXP = "E7_early_stopping"
FORCE_RERUN = False
RUN_TRAINING = FORCE_RERUN or not rg.already_done(cfg, EXP, ["early_stopping_by_seed.csv"])

X, y, feature_names = rg.load_ulb(cfg)
W, C = cfg["fusion_weights"], cfg["contamination"]

if RUN_TRAINING:
    run = rg.Run(cfg, EXP, "05_E7_early_stopping.ipynb", "Validation vs held-out early stopping for the AE")
    rows = []
    for seed in cfg["seeds"]:
        z = rg.load_scores(cfg, "E1_components", f"ulb_seed{seed}")
        tr, va, te = rg.get_split(cfg, "ulb", y, seed)
        assert np.array_equal(te, z["test_idx"]), "split differs from the one used in notebook 01"
        F = rg.prepare_features(X[tr], X[va], X[te], feature_names, seed, cfg)
        ae, secs, info = rg.fit_detectors(F, y[tr], seed, cfg, detectors=("AE",), ae_early_stopping="heldout")

        for protocol, ae_val, ae_test, stop_source in [
            ("Shared_Validation", z["AE_val"], z["AE_test"], "Validation partition (15%)"),
            ("Heldout_Legitimate_10pct", ae["AE"][1], ae["AE"][2], "10% of legitimate training rows"),
        ]:
            val_s = W["IF"] * z["IF_val"] + W["OCSVM"] * z["OCSVM_val"] + W["AE"] * ae_val
            test_s = W["IF"] * z["IF_test"] + W["OCSVM"] * z["OCSVM_test"] + W["AE"] * ae_test
            thr = rg.percentile_threshold(val_s, C)
            rows.append({"Seed": seed, "Protocol": protocol, "Early_Stop_Source": stop_source,
                         "Calibration_Source": "Validation partition (15%)",
                         **rg.evaluate(z["y_test"], test_s, thr, val_s)})
        print(f"seed {seed} done (held-out AE stopped after {info['AE_epochs']} epochs)")

    res = pd.DataFrame(rows)
    run.save_csv(res, "early_stopping_by_seed.csv")
    run.save_csv(rg.mean_sd(res, ["Protocol"], rg.METRICS + ["TP", "FP"]), "early_stopping_summary.csv")
    run.finish()

display(rg.read_result(cfg, EXP, "early_stopping_summary.csv")[["Protocol", "F1_mean", "F1_sd", "PR_AUC_mean", "PR_AUC_sd"]].round(4))